In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score

In [2]:
data = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'

In [3]:
!wget $data -O course_lead_scoring_2026.csv

--2026-09-28 08:03:25--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv


Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 278746 (272K) [text/plain]
Saving to: ‘course_lead_scoring_2026.csv’

course_lead_scoring 100%[===================>] 272.21K  --.-KB/s    in 0.005s  

2026-09-28 08:03:26 (54.1 MB/s) - ‘course_lead_scoring_2026.csv’ saved [278746/278746]



In [4]:
df = pd.read_csv('course_lead_scoring_2026.csv')

In [5]:
df

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1
...,...,...,...,...,...,...,...,...,...
4995,organic_search,education,employed,asia,45834.0,2,2,0.37,1
4996,NaN,technology,employed,north_america,84831.0,2,5,0.51,0
4997,organic_search,finance,employed,asia,53636.0,4,6,0.75,1
4998,referral,technology,employed,africa,69430.0,4,8,0.79,1


In [6]:
df.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [7]:
df.columns=df.columns.str.replace(' ','_').str.lower()

In [8]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [9]:
num_cols = df.select_dtypes(include=['number']).columns.drop(labels=['converted']).to_list()
num_cols

['annual_income',
 'number_of_courses_viewed',
 'interaction_count',
 'lead_score']

In [10]:
df[num_cols]

,annual_income,number_of_courses_viewed,interaction_count,lead_score
0,88160.0,3,4,0.64
1,72688.0,3,7,0.72
2,44697.0,3,4,0.58
3,NaN,3,6,0.57
4,24062.0,4,5,0.62
...,...,...,...,...
4995,45834.0,2,2,0.37
4996,84831.0,2,5,0.51
4997,53636.0,4,6,0.75
4998,69430.0,4,8,0.79


In [11]:
df[num_cols] = df[num_cols].fillna(0.0)

In [12]:
cat_cols = df.select_dtypes(exclude=['number']).columns.to_list()
cat_cols

['lead_source', 'industry', 'employment_status', 'location']

In [13]:
# for col in cat_cols:
#    df[col] = df[col].str.lower().str.replace(' ', '_')

In [14]:
df[cat_cols] = df[cat_cols].fillna('NA')

In [15]:
df['industry'].describe()

count           5000
unique             7
top       technology
freq            1173
Name: industry, dtype: object

In [16]:
df_full_train, df_test = train_test_split(df,test_size=0.2,random_state=42)
df_train, df_val = train_test_split(df_full_train,test_size=0.25,random_state=42)

In [17]:
len(df_train),len(df_val), len(df_test)

(3000, 1000, 1000)

In [18]:
df_full_train = df_full_train.reset_index(drop=True)
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [19]:
y_full_train = df_full_train.converted.values
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values
y_train

array([1, 0, 1, ..., 1, 0, 0], shape=(3000,))

In [20]:
df_train = df_train.drop(columns=['converted'])
df_val = df_val.drop(columns=['converted'])
df_test = df_test.drop(columns=['converted'])

In [21]:
df_full_train[num_cols].corrwith(df_full_train.converted)

annual_income               0.188437
number_of_courses_viewed    0.393200
interaction_count           0.450273
lead_score                  0.482078
dtype: float64

In [22]:
df_full_train[['interaction_count']].corrwith(df_full_train.lead_score)

interaction_count    0.916626
dtype: float64

In [23]:
df_full_train[['number_of_courses_viewed']].corrwith(df_full_train.lead_score)

number_of_courses_viewed    0.757178
dtype: float64

In [24]:
df_full_train[['number_of_courses_viewed']].corrwith(df_full_train.interaction_count)

number_of_courses_viewed    0.72304
dtype: float64

In [25]:
df_full_train[['annual_income']].corrwith(df_full_train.interaction_count)

annual_income    0.122528
dtype: float64

In [26]:
df_full_train.converted.value_counts(normalize=True)

converted
1    0.57075
0    0.42925
Name: proportion, dtype: float64

In [27]:
global_converted = df_full_train.converted.mean()
global_converted

np.float64(0.57075)

In [28]:
lead_score_converted = df_full_train.lead_score.mean()
interaction_count_converted = df_full_train.interaction_count.mean()
number_of_courses_viewed_converted = df_full_train.number_of_courses_viewed.mean()
annual_income_converted = df_full_train.annual_income.mean()

In [29]:
def mutual_info_converted_score(series):
    return round(mutual_info_score(series, df_full_train.converted),2)

In [30]:
mi = df_full_train[cat_cols].apply(mutual_info_converted_score)
mi.sort_values(ascending=False)

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

In [31]:
train_dicts = df_train[cat_cols + num_cols].to_dict(orient='records')
val_dicts = df_val[cat_cols + num_cols].to_dict(orient='records')


In [32]:
from sklearn.feature_extraction import DictVectorizer

In [33]:
dv = DictVectorizer(sparse=False)

In [34]:
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

In [35]:
X_train.shape

(3000, 28)

In [36]:
dv.get_feature_names_out()

array(['annual_income', 'employment_status=NA',
       'employment_status=employed', 'employment_status=self_employed',
       'employment_status=student', 'employment_status=unemployed',
       'industry=NA', 'industry=education', 'industry=finance',
       'industry=healthcare', 'industry=manufacturing', 'industry=retail',
       'industry=technology', 'interaction_count', 'lead_score',
       'lead_source=NA', 'lead_source=events',
       'lead_source=organic_search', 'lead_source=paid_ads',
       'lead_source=referral', 'lead_source=social_media', 'location=NA',
       'location=africa', 'location=asia', 'location=europe',
       'location=north_america', 'location=south_america',
       'number_of_courses_viewed'], dtype=object)

In [37]:
from sklearn.linear_model import LogisticRegression

In [38]:
y_train

array([1, 0, 1, ..., 1, 0, 0], shape=(3000,))

In [39]:
X_train

array([[6.5980e+04, 0.0000e+00, 1.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        2.0000e+00],
       [7.1662e+04, 0.0000e+00, 1.0000e+00, ..., 0.0000e+00, 1.0000e+00,
        0.0000e+00],
       [4.0220e+04, 0.0000e+00, 1.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        2.0000e+00],
       ...,
       [7.1532e+04, 0.0000e+00, 1.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        3.0000e+00],
       [4.8836e+04, 0.0000e+00, 1.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        0.0000e+00],
       [6.8969e+04, 0.0000e+00, 1.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        0.0000e+00]], shape=(3000, 28))

In [40]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass setting one can wrap it with the :class:`~sklearn.multiclass.OneVsRestClassifier`... warning:: The choice of the algorithm depends on the penalty chosen (`l1_ratio=0` for L2-penalty, `l1_ratio=1` for L1-penalty and `0 < l1_ratio < 1` for Elastic-Net) and on (multinomial) multiclass support: ================= ======================== ====================== solver l1_ratio multinomial multiclass ================= ======================== ====================== 'lbfgs' l1_ratio=0 yes 'liblinear' l1_ratio=1 or l1_ratio=0 no 'newton-cg' l1_ratio=0 yes 'newton-cholesky' l1_ratio=0 yes 'sag' l1_ratio=0 yes 'saga' 0<=l1_ratio<=1 yes ================= ======================== ======================.. note:: 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`... seealso:: Refer to the :ref:`User Guide <Logistic_regression>` for more information regarding :class:`LogisticRegression` and more specifically the :ref:`Table <logistic_regression_solvers>` summarizing solver/penalty supports... versionadded:: 0.17 Stochastic Average Gradient (SAG) descent solver. Multinomial support in version 0.18... versionadded:: 0.19 SAGA solver... versionchanged:: 0.22 The default solver changed from 'liblinear' to 'lbfgs' in 0.22... versionadded:: 1.2 newton-cholesky solver. Multinomial support in version 1.6.",'liblinear'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The 

In [41]:
model.intercept_[0]

np.float64(-0.010838978497809095)

In [42]:
model.coef_[0].round(3)

array([-0.   , -0.001,  0.007, -0.003, -0.008, -0.006, -0.   , -0.003,
       -0.001, -0.002, -0.002, -0.002, -0.001,  0.137,  0.011,  0.   ,
       -0.009,  0.007, -0.014,  0.009, -0.004,  0.   , -0.002, -0.002,
       -0.001, -0.003, -0.004,  0.054])

In [43]:
y_pred = model.predict_proba(X_val)[:,1]
y_pred

array([0.61725691, 0.44364108, 0.60433996, 0.64562246, 0.52214159,
       0.77885722, 0.60698076, 0.54481903, 0.76770971, 0.52765495,
       0.75038798, 0.6470054 , 0.56857815, 0.51044645, 0.59921425,
       0.46620745, 0.62786484, 0.69167143, 0.69082953, 0.59101344,
       0.49964695, 0.55590822, 0.65303933, 0.6079306 , 0.7231576 ,
       0.51616805, 0.56141276, 0.57206269, 0.67043904, 0.55599224,
       0.63613322, 0.70337215, 0.45463513, 0.61897288, 0.65250689,
       0.42775649, 0.63943096, 0.62315373, 0.6602988 , 0.51736304,
       0.53774147, 0.76074014, 0.48326071, 0.74151288, 0.63662375,
       0.41194601, 0.45371292, 0.6417333 , 0.43851314, 0.87060984,
       0.72468973, 0.75643941, 0.5296636 , 0.66736678, 0.50444495,
       0.57644782, 0.72318139, 0.72497254, 0.6507741 , 0.628958  ,
       0.7217776 , 0.59231588, 0.70824617, 0.61451576, 0.49691434,
       0.74445314, 0.52773381, 0.61402574, 0.55979831, 0.59024431,
       0.69368703, 0.72962242, 0.76111566, 0.60744404, 0.52501

In [44]:
converted_decision = (y_pred>=0.5)

In [45]:
df_pred = pd.DataFrame()
df_pred['probability'] = y_pred
df_pred['prediction'] = converted_decision.astype(int)
df_pred['actual'] = y_val
df_pred['correct'] = df_pred.prediction == df_pred.actual

In [46]:
df_pred

,probability,prediction,actual,correct
0,0.617257,1,1,True
1,0.443641,0,1,False
2,0.604340,1,0,False
3,0.645622,1,1,True
4,0.522142,1,0,False
...,...,...,...,...
995,0.684710,1,1,True
996,0.669580,1,1,True
997,0.458473,0,0,True
998,0.561770,1,0,False


In [47]:
accuracy = round(df_pred.correct.mean(), 2)
accuracy

np.float64(0.64)

In [48]:
y_train

array([1, 0, 1, ..., 1, 0, 0], shape=(3000,))

In [55]:
features = cat_cols + num_cols
diffs = {}

train_dicts = df_train[features].to_dict(orient='records')
val_dicts = df_val[features].to_dict(orient='records')

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)
baseline_acc = (y_val == y_pred).mean()

for feature in features:
    selected = [f for f in features if f != feature]
    
    train_dicts_sel = df_train[selected].to_dict(orient='records')
    val_dicts_sel = df_val[selected].to_dict(orient='records')

    dv_sel = DictVectorizer(sparse=False)
    X_train_sel = dv_sel.fit_transform(train_dicts_sel)
    X_val_sel = dv_sel.transform(val_dicts_sel)

    model_sel = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model_sel.fit(X_train_sel, y_train)

    y_pred_sel = model_sel.predict_proba(X_val_sel)[:, 1]
    converted_decision_sel = (y_pred_sel >= 0.5)
    # converted_decision_sel = model_sel.predict(X_val_sel)
    
    accuracy_sel = (converted_decision_sel.astype(int) == y_val).mean()
    diffs[feature] = baseline_acc - accuracy_sel

for f, diff in diffs.items():
    print(f"{f}: {diff:.4f}")

lead_source: 0.0030
industry: 0.0000
employment_status: 0.0000
location: 0.0000
annual_income: -0.0790
number_of_courses_viewed: 0.0020
interaction_count: 0.0440
lead_score: 0.0010


In [58]:
c_values = [0.000001, 0.00001, 0.0001, 0.001]

features = cat_cols + num_cols
diffs = {}

train_dicts = df_train[features].to_dict(orient='records')
val_dicts = df_val[features].to_dict(orient='records')

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

for c in c_values:
  model = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
  model.fit(X_train, y_train)

  y_pred = model.predict(X_val)
  acc = round((y_val == y_pred).mean(), 3)
  print(f"c: {c} accuracy: {acc}")

c: 1e-06 accuracy: 0.598
c: 1e-05 accuracy: 0.598
c: 0.0001 accuracy: 0.613
c: 0.001 accuracy: 0.645
